In [ ]:
import tensorflow as tf
import numpy as np
import scipy.io.wavfile as wav
from pathlib import Path

In [ ]:
# Download Version 2 of the Speech Commands Dataset
!wget http://download.tensorflow.org/data/speech_commands_v0.02.tar.gz

# Create a folder and unzip (this takes ~2 mins)
!mkdir dataset
!tar -xzf speech_commands_v0.02.tar.gz -C dataset

--2026-02-21 08:28:39--  http://download.tensorflow.org/data/speech_commands_v0.02.tar.gz
Resolving download.tensorflow.org (download.tensorflow.org)... 74.125.199.207, 142.251.188.207, 74.125.20.207, ...
Connecting to download.tensorflow.org (download.tensorflow.org)|74.125.199.207|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2428923189 (2.3G) [application/gzip]
Saving to: ‘speech_commands_v0.02.tar.gz’

speech_commands_v0. 100%[===================>]   2.26G  59.0MB/s    in 22s     

2026-02-21 08:29:01 (104 MB/s) - ‘speech_commands_v0.02.tar.gz’ saved [2428923189/2428923189]



In [ ]:
def process_audio_to_model_input(audio_array, tflite_model_path):
    # 1. Load the TFLite model to get the exact quantization parameters
    interpreter = tf.lite.Interpreter(model_path=tflite_model_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()[0]
    scale, zero_point = input_details['quantization']
    print(scale, zero_point)

    # Ensure audio is exactly 16000 samples and float32 [-1.0, 1.0]
    audio_tensor = tf.convert_to_tensor(audio_array, dtype=tf.float32)

    # 2. Compute the Short-Time Fourier Transform (STFT)
    stfts = tf.signal.stft(
        audio_tensor,
        frame_length=640, # 40ms
        frame_step=320,   # 20ms
        fft_length=1024
    )
    spectrograms = tf.abs(stfts)

    # 3. Create Mel Filterbank (20 Hz to 4000 Hz)
    num_spectrogram_bins = stfts.shape[-1]
    linear_to_mel_weight_matrix = tf.signal.linear_to_mel_weight_matrix(
        num_mel_bins=40,
        num_spectrogram_bins=num_spectrogram_bins,
        sample_rate=16000,
        lower_edge_hertz=20.0,
        upper_edge_hertz=4000.0
    )

    mel_spectrograms = tf.tensordot(spectrograms, linear_to_mel_weight_matrix, 1)
    mel_spectrograms.set_shape(spectrograms.shape[:-1].concatenate(linear_to_mel_weight_matrix.shape[-1:]))

    # 4. Compute Log Mel & MFCCs (keep first 10 coefficients)
    log_mel_spectrograms = tf.math.log(mel_spectrograms + 1e-6)
    mfccs = tf.signal.mfccs_from_log_mel_spectrograms(log_mel_spectrograms)[..., :10]

    # 5. Quantize to int8 using the model's scale and zero_point
    mfccs_float = mfccs.numpy()

    # Apply standard asymmetric quantization formula
    mfccs_quantized = np.round((mfccs_float / scale) + zero_point)
    mfccs_quantized = np.clip(mfccs_quantized, -128, 127).astype(np.int8)

    # 6. Reshape to match CNN input [1, 49, 10, 1]
    model_input = np.reshape(mfccs_quantized, (1, 49, 10, 1))

    return model_input

In [ ]:
def predict(audio):
    model_path = 'cnn_s_quantized.tflite'
    processed = process_audio_to_model_input(audio, model_path)
    interpreter = tf.lite.Interpreter(model_path)
    interpreter.allocate_tensors()
    processed = np.reshape(processed, (1, 490))
    interpreter.set_tensor(0, processed)
    interpreter.invoke()
    prediction_index = np.argmax(interpreter.get_tensor(20)[0])

    labels = ['silence', 'unknown', 'yes', 'no', 'up', 'down', 'left', 'right', 'on', 'off', 'stop', 'go']
    print(f"Model predicts: {labels[prediction_index]}")

In [ ]:
data_dir = Path('dataset')

yes_files = list(data_dir.glob('yes/*.wav'))
on_files = list(data_dir.glob('on/*.wav'))

f1 = yes_files[0]
f2 = on_files[0]

sample_rate, yes_audio = wav.read(str(f1))

# Normalize to float32 between -1.0 and 1.0 (required for MFCC)
yes_audio = yes_audio.astype('float32') / 32768.0

sample_rate, on_audio = wav.read(str(f2))

# Normalize to float32 between -1.0 and 1.0 (required for MFCC)
on_audio = on_audio.astype('float32') / 32768.0

predict(yes_audio)
predict(on_audio)

Loaded 0d90d8e1_nohash_2.wav at 16000Hz
1.07421875 102
Model predicts: yes
1.07421875 102
Model predicts: on


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [ ]:
def generate_c_header():
    # --- 1. Generate the Mel Weight Matrix ---
    # Shape: [513, 40]
    mel_matrix = tf.signal.linear_to_mel_weight_matrix(
        num_mel_bins=40,
        num_spectrogram_bins=513, # (FFT_LEN / 2) + 1
        sample_rate=16000,
        lower_edge_hertz=20.0,
        upper_edge_hertz=4000.0
    ).numpy()

    # --- 2. Generate the DCT Matrix ---
    # We pass an identity matrix through TensorFlow's MFCC function.
    # This mathematically isolates the exact transformation matrix TensorFlow applies.
    identity_matrix = tf.eye(40)
    dct_matrix_full = tf.signal.mfccs_from_log_mel_spectrograms(identity_matrix).numpy()

    # tf.signal computes over the last axis, so the result is transposed.
    # We transpose it back and slice only the first 10 coefficients we need.
    # Final Shape: [10, 40]
    dct_matrix = np.transpose(dct_matrix_full)[:10, :]

    # --- 3. Format as C Header ---
    with open("freq_params.h", "w") as f:
        f.write("#ifndef MODEL_WEIGHTS_H\n")
        f.write("#define MODEL_WEIGHTS_H\n\n")

        # Write Mel Matrix
        f.write("// Shape: [513][40]\n")
        f.write("const float mel_weight_matrix[513][40] = {\n")
        for row in mel_matrix:
            f.write("    {" + ", ".join([f"{val:.6f}f" for val in row]) + "},\n")
        f.write("};\n\n")

        # Write DCT Matrix
        f.write("// Shape: [10][40]\n")
        f.write("const float dct_matrix[10][40] = {\n")
        for row in dct_matrix:
            f.write("    {" + ", ".join([f"{val:.6f}f" for val in row]) + "},\n")
        f.write("};\n\n")

        f.write("#endif // MODEL_WEIGHTS_H\n")

    print("Successfully generated freq_params.h")

generate_c_header()

Successfully generated freq_params.h


In [ ]:
def numpy_to_c_array(arr):
    """Recursively formats a NumPy array into a nested C-style string."""
    if arr.ndim == 1:
        return "{" + ", ".join(map(str, arr)) + "}"
    else:
        return "{" + ", ".join(numpy_to_c_array(sub) for sub in arr) + "}"

def get_c_dims(shape):
    """Converts a Python shape tuple into C array dimension brackets."""
    return "".join(f"[{d}]" for d in shape)

def export_test_data_to_c(raw_audio_int16, expected_model_input_int8):
    """
    Dumps the raw audio and the expected int8 tensor into a C header file.
    """
    with open("test_data.h", "w") as f:
        f.write("#ifndef TEST_DATA_H\n")
        f.write("#define TEST_DATA_H\n\n")
        f.write("#include <stdint.h>\n\n")

        # --- Write Raw Audio (Input) ---
        s = numpy_to_c_array(raw_audio_int16)
        dims = get_c_dims(raw_audio_int16.shape)
        f.write(f"const int16_t test_raw_audio{dims} = {s};\n\n")

        # --- Write Expected Output ---
        s = numpy_to_c_array(expected_model_input_int8)
        dims = get_c_dims(expected_model_input_int8.shape)
        f.write(f"const int8_t expected_model_input{dims} = {s};\n\n")

        f.write("#endif // TEST_DATA_H\n")

    print("Successfully generated test_data.h")

sample_rate, original_audio = wav.read(str(f2))
float_audio = original_audio.astype('float32') / 32768.0
expected_output = process_audio_to_model_input(float_audio, "cnn_s_quantized.tflite")
export_test_data_to_c(original_audio, expected_output)

1.07421875 102
Successfully generated test_data.h


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
